In [ ]:
import sys
import os
from scipy.spatial import cKDTree

chemin_lib = os.path.abspath("..")

# Add path to Python path
sys.path.append(chemin_lib)

import pandas as pd

from sklearn.preprocessing import MultiLabelBinarizer
import torch
import numpy as np
import SimpleITK as sitk 

from metriclib.metric import StreamMetric, MetricResult, TabularMetric
from metriclib.report import Report

from metriclib.data import Dataset

In [ ]:
### CHAOSDataset
    
class CHAOSDataset(Dataset):
    def __init__(self):
        base_dir = os.path.join("..", "sample-data", "CHAOS_dataset")
        df = pd.read_csv(os.path.join(base_dir, "CHAOS_metadatas.csv"))
        self.metadata = df
        self.metadata["CHAOS_ID"] = self.metadata["CHAOS_ID"].astype(int)
        self.labels = self.metadata["CHAOS_ID"]

    def __len__(self):
        return len(self.metadata.index)

    def __getitem__(self, index):
        metadata = self.metadata.iloc[index].to_dict()
        metadata["idx"] = int(index)
        num_id = str(self.metadata["CHAOS_ID"].iloc[index])
        record_path = os.path.join("..", "sample-data", "CHAOS_dataset", "img_NIFTI", "CHAOS_"+str(num_id)+".nii.gz") # Need to put the right files in the "img_NIFTI" folder
        
        if os.path.isfile(record_path):
            # x : mettre un seul tensor
            img = sitk.ReadImage(record_path)
            
            x = torch.tensor(sitk.GetArrayFromImage(img))
        else : 
            print("Input image file not found, initializing with x=1 .")
            x = 1 # to remplace with the "img_NIFTI" files
            
        
        seg1_path = os.path.join("..", "sample-data", "CHAOS_dataset", "seg_TotalSegmentator_NIFTI", "CHAOS_"+str(num_id)+".nii.gz")
        img_seg1 = sitk.ReadImage(seg1_path)
        seg1 = sitk.GetArrayFromImage(img_seg1)

        seg2_path = os.path.join("..", "sample-data", "CHAOS_dataset", "seg_nnInteractive_NIFTI", "CHAOS_"+str(num_id)+".nii.gz")
        img_seg2 = sitk.ReadImage(seg2_path)
        seg2 = sitk.GetArrayFromImage(img_seg2)

        y = torch.tensor([seg1, seg2])
                
        return x, y, metadata

dataset_chaos = CHAOSDataset()

In [3]:
report = Report(datasets=[dataset_chaos])
# mapping for columns in metadata file
dict_config = {'seg1_origin' : 'seg_nnInteractive_origin', 
               'seg1_spacing':'seg_nnInteractive_spacing', 
               'seg1_direction':'seg_nnInteractive_direction',
               'seg2_origin' : 'seg_TotalSegmentator_origin', 
               'seg2_spacing':'seg_TotalSegmentator_spacing', 
               'seg2_direction':'seg_TotalSegmentator_direction'}

report.add_metric("DICESimilarityCoefficient", metric_config=dict_config)
report.add_metric("IntersectionOverUnion", metric_config=dict_config)
report.add_metric("HausdorffDistance", metric_config=dict_config)
report.add_metric("HausdorffDistance95", metric_config=dict_config)
test = report.generate()

  0%|          | 0/4 [00:00<?, ?it/s]/tmp/ipykernel_2889579/3752299182.py:38: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  y = torch.tensor([seg1, seg2])
100%|██████████| 4/4 [01:46<00:00, 26.57s/it]


In [4]:
test

([{'metric': metriclib.metrics.measurement_process.DICESimilarityCoefficient,
   'reference': None,
   'metric_config': {'seg1_origin': 'seg_nnInteractive_origin',
    'seg1_spacing': 'seg_nnInteractive_spacing',
    'seg1_direction': 'seg_nnInteractive_direction',
    'seg2_origin': 'seg_TotalSegmentator_origin',
    'seg2_spacing': 'seg_TotalSegmentator_spacing',
    'seg2_direction': 'seg_TotalSegmentator_direction'},
   'dataset': 'CHAOSDataset',
   'name': None,
   'result': MetricResult(description='DICE Score between two segmentations', value=[0.9557618292959021, 0.954658851454453, 0.9496780735107732, 0.9502340455114242], cluster=None, threshold=0)},
  {'metric': metriclib.metrics.measurement_process.IntersectionOverUnion,
   'reference': None,
   'metric_config': {'seg1_origin': 'seg_nnInteractive_origin',
    'seg1_spacing': 'seg_nnInteractive_spacing',
    'seg1_direction': 'seg_nnInteractive_direction',
    'seg2_origin': 'seg_TotalSegmentator_origin',
    'seg2_spacing': 's